In [115]:
import transformers
from tokenizers import Tokenizer

## Load training data

In [116]:
train_files = ["/srv/data/lt2326-h26/a1/train/en.txt", 
                "/srv/data/lt2326-h26/a1/train/tr.txt", 
                "/srv/data/lt2326-h26/a1/train/zh.txt"]

## Analyze unique character

In [117]:
chars = set()
for file in train_files:
    with open(file, encoding="utf-8") as f:
        for line in f:
            chars.update(line.strip())

print(len(chars))

9442


## Train BPE tokenizers

In [118]:
from tokenizers.trainers import BpeTrainer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import UnicodeScripts

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = UnicodeScripts()

### Small BPE tokenizer

In [119]:
small_trainer = BpeTrainer(vocab_size=12000, special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])

In [120]:
tokenizer.train(train_files, small_trainer)
tokenizer.save("trained_tokenizers/small_bpe_tokenizer.json")

### Large BPE tokenizer

In [121]:
big_trainer = BpeTrainer(vocab_size=20000, special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])

In [122]:
tokenizer.train(train_files, big_trainer)
tokenizer.save("trained_tokenizers/large_bpe_tokenizer.json")

### Train a character-based tokenizer

In [123]:
from tokenizers.trainers import WordLevelTrainer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Split
from tokenizers import Regex

tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Split(Regex("."), behavior="isolated") #split on unicode character and make every character its own token ("isolated")

In [124]:
char_trainer = WordLevelTrainer(vocab_size=10000, special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])

In [125]:
tokenizer.train(train_files, char_trainer)
tokenizer.save("trained_tokenizers/char_tokenizer.json")

## Analyze the tokenizers

In [126]:
small_bpe_tokenizer = Tokenizer.from_file("trained_tokenizers/small_bpe_tokenizer.json")

In [127]:
output = small_bpe_tokenizer.encode("coding at this ungodly hour?")
print(output.tokens)

['co', 'd', 'ing ', 'at ', 'this ', 'un', 'g', 'od', 'ly ', 'h', 'our', '?']


In [128]:
large_bpe_tokenizer = Tokenizer.from_file("trained_tokenizers/large_bpe_tokenizer.json")

In [129]:
output = large_bpe_tokenizer.encode("coding at this ungodly hour?")
print(output.tokens)

['co', 'd', 'ing ', 'at ', 'this ', 'ung', 'od', 'ly ', 'hour', '?']


In [130]:
char_tokenizer = Tokenizer.from_file("trained_tokenizers/char_tokenizer.json")

In [131]:
print(char_tokenizer.token_to_id("[UNK]")) 

0


In [132]:
output = char_tokenizer.encode("coding at this ungodly hour?")
print(output.tokens)

['c', 'o', 'd', 'i', 'n', 'g', ' ', 'a', 't', ' ', 't', 'h', 'i', 's', ' ', 'u', 'n', 'g', 'o', 'd', 'l', 'y', ' ', 'h', 'o', 'u', 'r', '?']


### Validation set

In [133]:
val_files = ["/srv/data/lt2326-h26/a1/valid/en.txt", 
            "/srv/data/lt2326-h26/a1/valid/tr.txt", 
            "/srv/data/lt2326-h26/a1/valid/zh.txt"]


tokenizers = {"small_bpe_tokenizer" : small_bpe_tokenizer,
              "large_bpe_tokenizer" : large_bpe_tokenizer,
              "char_tokenizer": char_tokenizer}

In [134]:
val_lines = {}
val_encodings = {}
file_unicode_chars = {}

# Encoding(num_tokens=99, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

for key,value in tokenizers.items():
    print("Tokenizer: ", key)
    print("Vocab size: ", value.get_vocab_size())
    for file in val_files:
        with open(f"{file}", encoding="utf-8") as f:
            val_lines[file] = [line.strip() for line in f if line.strip()]
        file_unicode_chars[file] = sum(len(line) for line in val_lines[file])
        val_encodings[file] = value.encode_batch(val_lines[file], add_special_tokens=False)
    
    for file in val_encodings:
        print(file)
        num_sent = len(val_encodings[file])
        total_num_tokens = 0
        for i in range(len(val_encodings[file])):
            num_tokens = len(val_encodings[file][i])
            total_num_tokens += num_tokens
        print("Total num tokens: ", total_num_tokens)
        print("Avg num tokens/sent: ", total_num_tokens/num_sent)
        print("Avg num chars/token: ", file_unicode_chars[file]/total_num_tokens)
    print("-----------------------------------------------------------------------")

Tokenizer:  small_bpe_tokenizer
Vocab size:  12000
/srv/data/lt2326-h26/a1/valid/en.txt
Total num tokens:  134355
Avg num tokens/sent:  29.59361233480176
Avg num chars/token:  2.7410144765732576
/srv/data/lt2326-h26/a1/valid/tr.txt
Total num tokens:  138104
Avg num tokens/sent:  36.9756358768407
Avg num chars/token:  2.6674317905346694
/srv/data/lt2326-h26/a1/valid/zh.txt
Total num tokens:  312367
Avg num tokens/sent:  32.694892191752146
Avg num chars/token:  1.1789529623807893
-----------------------------------------------------------------------
Tokenizer:  large_bpe_tokenizer
Vocab size:  20000
/srv/data/lt2326-h26/a1/valid/en.txt
Total num tokens:  104461
Avg num tokens/sent:  23.009030837004406
Avg num chars/token:  3.525420970505739
/srv/data/lt2326-h26/a1/valid/tr.txt
Total num tokens:  106742
Avg num tokens/sent:  28.578848728246317
Avg num chars/token:  3.4511532480185867
/srv/data/lt2326-h26/a1/valid/zh.txt
Total num tokens:  273527
Avg num tokens/sent:  28.629579233828764
A

### Examine results

In [135]:
for tokenizer_name,tokenizer in tokenizers.items():
    print("Tokenizer: ", tokenizer_name)
    for file in val_lines:
        last_three = val_lines[file][-3:]
        print("File: ", file)
        for sent in last_three:
            print("  ", sent)
            print("Tokens: ", tokenizer.encode(sent, add_special_tokens=False).tokens)
    print("--------------------------------------------------------")

Tokenizer:  small_bpe_tokenizer
File:  /srv/data/lt2326-h26/a1/valid/en.txt
   Some of her writing is also for children.
Tokens:  ['Some ', 'of ', 'her ', 'writ', 'ing ', 'is ', 'also ', 'for ', 'child', 'ren', '.']
   He became a sports superstar known around the world.
Tokens:  ['He ', 'became ', 'a s', 'por', 'ts ', 'su', 'per', 'st', 'ar ', 'known ', 'around ', 'the ', 'world', '.']
   Orbital resonances and chaos in the Solar System.
Tokens:  ['Or', 'bit', 'al ', 'res', 'on', 'anc', 'es and ', 'ch', 'a', 'os ', 'in the ', 'S', 'ol', 'ar ', 'S', 'y', 'st', 'em', '.']
File:  /srv/data/lt2326-h26/a1/valid/tr.txt
   Ancak AH-64D bunun yerine o yıl seçildi.
Tokens:  ['Ancak ', 'A', 'H', '-', '6', '4', 'D ', 'b', 'unun ', 'yer', 'ine ', 'o ', 'yıl ', 'seç', 'ildi', '.']
   Sunucu, muhabir Metin Tok'a bağlanırken ses uydudan geç gelir.
Tokens:  ['S', 'un', 'uc', 'u', ', ', 'mu', 'ha', 'bir ', 'M', 'eti', 'n ', 'T', 'ok', "'", 'a ', 'bağ', 'lan', 'ır', 'ken ', 's', 'es ', 'uy', 'du', 'dan